In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
catalog = "ete"
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"
data_source = "products"

df_bronze = spark.table(f"{catalog}.{bronze_schema}.{data_source}")

print("Raw Bronze Products:")
display(df_bronze)

In [0]:
df_silver = (
    df_bronze
    .dropDuplicates(['product_id'])
    
    .withColumn("category", F.when(F.col("category").isNull(), None).otherwise(F.initcap("category")))
    
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("product_name", F.regexp_replace(F.col("product_name"), "(?i)Coolng", "Cooling"))
    .withColumn("product_name", F.regexp_replace(F.col("product_name"), "(?i)Hedphone", "Headphone"))
    .withColumn("product_name", F.regexp_replace(F.col("product_name"), "(?i)Keybord", "Keyboard"))
    .withColumn("product_name", F.regexp_replace(F.col("product_name"), "(?i)Mausepad", "Mousepad"))
)

In [0]:
df_silver = (
    df_bronze
    .dropDuplicates(['product_id'])
    
    .withColumn("category", F.when(F.col("category").isNull(), None).otherwise(F.initcap("category")))
    
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("product_name", F.regexp_replace(F.col("product_name"), "(?i)Coolng", "Cooling"))
    .withColumn("product_name", F.regexp_replace(F.col("product_name"), "(?i)Hedphone", "Headphone"))
    .withColumn("product_name", F.regexp_replace(F.col("product_name"), "(?i)Keybord", "Keyboard"))
    .withColumn("product_name", F.regexp_replace(F.col("product_name"), "(?i)Mausepad", "Mousepad"))
)

df_silver = (
    df_silver
    .withColumn(
        "division",
        F.when(F.col("category") == "Cooling", "Internal Components")
         .when(F.col("category") == "Audio", "Peripherals")
         .when(F.col("category") == "Peripherals", "Peripherals")
         .when(F.col("category") == "Lighting", "Accessories")
         .otherwise("Other")
    )
    
    .withColumn("variant", F.substring_index(F.col("product_name"), " ", -1))
    
    .withColumnRenamed("product_id", "product_code")
    
    .withColumnRenamed("product_name", "product")
)

df_silver = df_silver.select("product_code", "division", "category", "product", "variant", "read_timestamp", "file_name", "file_size")

display(df_silver)

In [0]:
silver_table_name = f"{catalog}.{silver_schema}.{data_source}"

df_silver.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(silver_table_name)

print(f" Silver products successfully written to {silver_table_name}")

Gold Processing....

In [0]:
df_gold=df_silver.select("product_code", "division", "category", "product", "variant")
child_gold_table = f"{catalog}.{gold_schema}.sb_dim_{data_source}"
df_gold.show(5)

In [0]:
child_gold_table = f"{catalog}.{gold_schema}.sb_dim_{data_source}"

df_gold.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(child_gold_table)

In [0]:
parent_table_name = f"{catalog}.{gold_schema}.dim_products"
delta_table = DeltaTable.forName(spark, parent_table_name)
df_child_products = spark.table(child_gold_table)

In [0]:
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).execute()

print(" Products perfectly matched and merged!")